In [1]:
# --- 1. Setup and Installations ---
# Upgrade pip to ensure compatibility
!pip install --upgrade pip

# Install required libraries
!pip install ultralytics roboflow gradio opencv-python-headless matplotlib numpy wget

# Download YOLOv10n pre-trained weights
# This will save the model to /content/yolov10n.pt in Colab
# If the file already exists from a previous run, wget will skip downloading again.
!wget -nc -O /content/yolov10n.pt https://github.com/jameslahm/yolov10/releases/download/v1.0/yolov10n.pt

# --- 2. Roboflow Dataset Download ---
# IMPORTANT: Do NOT share your API key publicly.
# If running this notebook, replace "YOUR_ROBOFLOW_API_KEY" with your actual API key.
# For better security, especially in shared environments, you can set it as an environment variable
# via Colab Secrets (left sidebar, padlock icon) or directly in a code cell before this block.
# Example using environment variable:
# import os
# os.environ["ROBOFLOW_API_KEY"] = "YOUR_ROBOFLOW_API_KEY_HERE" # Replace with your key

from roboflow import Roboflow
import os

# Get API key from environment variable (recommended for security) or set directly here
# If using Colab Secrets, it might be named "ROBOFLOW_API_KEY"
ROBOFLOW_KEY = os.environ.get("ROBOFLOW_API_KEY", "nhmwq3dX2803KVRmQ22h") # REPLACE with your actual Roboflow API Key if not using env var

if ROBOFLOW_KEY == "nhmwq3dX2803KVRmQ22h":
    print("WARNING: Using a placeholder API key. Please replace it with your actual Roboflow API key or set the ROBOFLOW_API_KEY environment variable.")
    print("If you intend to use this notebook publicly, consider using Colab Secrets for your API key.")

rf = Roboflow(api_key=ROBOFLOW_KEY)

# Replace 'clg-vtj9f' and 'blood-cell-detection-bsbvn' with your actual workspace and project IDs if different.
# Make sure 'version(4)' matches the version you intend to use.
project = rf.workspace("clg-vtj9f").project("blood-cell-detection-bsbvn")
version = project.version(4)

print("Downloading dataset... This may take a while.")
# Download the dataset in YOLOv9 format (compatible with YOLOv10)
# This will download to /content/blood-cell-detection-4/
dataset = version.download("yolov9")
print(f"Dataset downloaded successfully to {dataset.location}")


# --- 3. Train the YOLOv10 Model ---
from ultralytics import YOLO
import wandb # Optional: For Weights & Biases logging

# Optional: Log in to Weights & Biases if you want to track training metrics
# Make sure to set your WANDB_API_KEY environment variable (e.g., in Colab Secrets)
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")
if WANDB_API_KEY:
    print("Logging into Weights & Biases...")
    wandb.login(key=WANDB_API_KEY)
else:
    print("WANDB_API_KEY not found. Training will proceed without W&B logging.")

# Load the pre-trained YOLOv10n model
model = YOLO("/content/yolov10n.pt")

# Start training
# Results will be saved in /content/runs/detect/train_blood_cells/
print("Starting model training...")
model.train(
    data=os.path.join(dataset.location, "data.yaml"), # Path to your dataset config
    epochs=25,
    batch=32,
    imgsz=640, # Input image size for training
    plots=True, # Generate training plots (e.g., confusion matrix, F1 curve)
    project="/content/runs", # Where to save the training results
    name="detect/train_blood_cells" # Subdirectory name for this training run
)
print("Training complete. Model weights saved to /content/runs/detect/train_blood_cells/weights/best.pt")


# --- 4. Visualize Prediction Results (Optional) ---
# This section aims to display sample prediction images.
# After training, Ultralytics usually generates validation plots in the training run directory.

import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Path to the directory where training results (including plots) are saved
training_results_dir = "/content/runs/detect/train_blood_cells"

# Look for validation batch plots or prediction results
image_paths = []
# Option 1: Validation batch plots from training
image_paths.extend(glob.glob(os.path.join(training_results_dir, "val_batch*_pred.jpg")))
# Option 2: Prediction results if a 'predict' command was run separately after training
# You would uncomment and modify this if you run 'model.predict' on a test set separately
# image_paths.extend(glob.glob("/content/runs/detect/predict_test_set/*.jpg"))

if image_paths:
    images_to_display = image_paths[:10] # Display up to 10 images
    fig, axes = plt.subplots(2, 5, figsize=(20, 10))
    fig.suptitle('Sample Predicted Images', fontsize=16)

    for i, ax in enumerate(axes.flat):
        if i < len(images_to_display):
            img = mpimg.imread(images_to_display[i])
            ax.imshow(img)
            ax.axis('off') # Hide axes
        else:
            ax.axis('off') # Hide empty subplots
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make room for suptitle
    plt.show()
else:
    print(f"No images found for visualization in {training_results_dir} or other specified paths.")
    print("Ensure training completed successfully and generated plots, or run a separate 'predict' step.")


# --- 5. Gradio Web Application for Real-time Inference ---
import gradio as gr
import cv2
import numpy as np
from collections import Counter
from ultralytics import YOLO # Import YOLO for model loading

# Load the trained model for inference in the Gradio app
# Adjust this path to your actual best.pt file after training
# If training failed or you want to use the pre-trained, adjust accordingly
MODEL_PATH = "/content/runs/detect/train_blood_cells/weights/best.pt"
if not os.path.exists(MODEL_PATH):
    print(f"Warning: Trained model not found at {MODEL_PATH}. Using pre-trained yolov10n.pt for app.")
    model = YOLO("/content/yolov10n.pt") # Fallback to pre-trained
else:
    model = YOLO(MODEL_PATH)
    print(f"Loaded model from: {MODEL_PATH}")

def predict_image(image):
    if image is None:
        return None, "Please upload an image."

    # Convert input image to RGB (Gradio provides NumPy array in RGB)
    # Ultralytics model expects RGB. OpenCV reads images as BGR by default,
    # but Gradio passes numpy arrays in RGB, so conversion might be needed based on source.
    # For Gradio's numpy image input, it's typically already RGB.
    # If using cv2.imread somewhere, you'd need cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Run inference
    results = model(image, imgsz=640, conf=0.25, verbose=False) # verbose=False to suppress console output
    result = results[0] # Get the results for the first (and only) image

    # Plot annotations directly onto the image
    # result.plot() returns an image with annotations, typically in BGR format by default from Ultralytics
    annotated_img = result.plot()

    # Extract detections and count classes
    detections = result.boxes.data # Get tensor of detections
    
    detection_str = "No blood cells detected."
    if detections.numel() > 0: # Check if the tensor has any elements (i.e., detections)
        class_ids = detections[:, 5].cpu().numpy().astype(int) # Get class IDs (last column of data)
        # Ensure class names are obtained safely
        if hasattr(model, 'names') and model.names:
            class_names = [model.names[int(cls_id)] for cls_id in class_ids if int(cls_id) in model.names]
            if class_names:
                count = Counter(class_names) # Count occurrences of each class
                detection_str = ", ".join([f"{name}: {c}" for name, c in count.items()])
            else:
                detection_str = "Detected objects with unknown classes."
        else:
            detection_str = f"Detected {len(class_ids)} objects (class names not available)."
    
    # Convert annotated image from BGR to RGB for Gradio display (Gradio expects RGB)
    annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

    return annotated_img_rgb, detection_str

# Create and launch the Gradio interface
app = gr.Interface(
    predict_image,
    inputs=gr.Image(type="numpy", label="Upload a Microscope Image"),
    outputs=[
        gr.Image(type="numpy", label="Annotated Image"),
        gr.Textbox(label="Detection Count")
    ],
    title="Blood Cell Detection & Count",
    description="Upload a microscope image to detect various types of blood cells and get their counts. This application uses a YOLOv10 model.",
    # You can add example images here if you have them publicly accessible or in your Colab files
    # examples=[["path/to/your/example_image.jpg"]]
)

print("Launching Gradio app...")
# share=True generates a public URL for sharing (temporary, typically lasts 72 hours)
# debug=True provides more detailed error messages in the console
app.launch(debug=True, share=True)

Defaulting to user installation because normal site-packages is not writeable
  Using cached pip-25.1.1-py3-none-any.whl.metadata (3.6 kB)
Using cached pip-25.1.1-py3-none-any.whl (1.8 MB)


ERROR: To modify pip, please run the following command:
C:\Python313\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached ultralytics-8.3.137-py3-none-any.whl.metadata (37 kB)
  Using cached roboflow-1.1.64-py3-none-any.whl.metadata (9.7 kB)
  Using cached gradio-5.29.1-py3-none-any.whl.metadata (16 kB)
  Using cached opencv_python_headless-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached wget-3.2-py3-none-any.whl
  Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached torch-2.7.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached torchvision-0.22.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached psutil-7.0.0-cp37-abi3-win_amd64.whl.metadata (23 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
  Using cached idna-3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Usi

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

ModuleNotFoundError: No module named 'roboflow'